# ShiftScope

**Compare any two single-cell populations**, measure how far the cell-state
distribution shifts, localize *where*, find the *driver genes*, and let Claude
interpret the biology. Condition-agnostic: control vs perturbation, disease vs
healthy, young vs old, cell type vs cell type — one tool.

This notebook's worked example is **control vs IFN-β-stimulated monocytes**
(Kang 2018). It downloads a 38 MB dataset and runs in ~20 s on CPU.

In [ ]:
# === Colab setup — Runtime > Run all (skip this cell if running locally from a clone) ===
# The repo must be PUBLIC (Colab can't clone a private repo over https).
# Installs only what Colab lacks, PINNING numpy+pandas to Colab's existing versions so nothing
# is upgraded — that's what avoids the numpy ABI break that would otherwise need a restart.
# (If cell 2 ever still fails to import: Runtime > Restart session, then Run all again.)
![ -d /content/claude_science_hackathon ] || git clone -q https://github.com/aenorhabditis6/claude_science_hackathon.git /content/claude_science_hackathon
%cd /content/claude_science_hackathon
import numpy, pandas
!pip install -q "numpy=={numpy.__version__}" "pandas=={pandas.__version__}" scanpy anndata pot umap-learn leidenalg igraph s3fs h5py anthropic gradio 'typing_extensions>=4.15'
print('Setup done. Now: Runtime > Run all.  (No restart needed. If an import fails: Restart session, then Run all.)')

In [ ]:
import sys, os
sys.path.insert(0, '.')  # so `from shiftscope import ...` works from the repo root
# On Colab, pull the API key from Secrets (the 🔑 icon in the sidebar) if you added one named
# ANTHROPIC_API_KEY. Enables §7 + §8c Claude features; without it they degrade gracefully.
try:
    from google.colab import userdata
    os.environ.setdefault('ANTHROPIC_API_KEY', userdata.get('ANTHROPIC_API_KEY'))
except Exception:
    pass  # not on Colab, or no secret set
from shiftscope import io, embed, compare, localize, drivers, interpret
import scanpy as sc
sc.settings.verbosity = 1

## 1. Load — control vs IFN-β-stimulated monocytes (Kang 2018)

`load_public` fetches a ready-made dataset and tags a `(reference, query)` pair. The
`within=` argument first restricts to one subpopulation, so we compare ctrl vs stim
*inside* monocytes. Any local file works the same way via `io.load(path, key, ref, query)`.

In [ ]:
adata = io.load_public('kang', 'ctrl', 'stim', within=('cell_type', 'CD14+ Monocytes'))
print(adata)
adata.obs['_group'].value_counts()

## 2. Embed — one shared PCA on the pooled cells, plus a 2D UMAP

In [ ]:
Z_ref, Z_query = embed.embed(adata, n_pcs=30)
print('Z_ref', Z_ref.shape, ' Z_query', Z_query.shape)

## 3. Compare — E-distance + E-test (scPerturb), MMD, Sinkhorn-OT

The E-distance and its permutation E-test are the scPerturb / pertpy standard for how
strongly a perturbation shifts a population. Expect a large distance and a tiny p-value.

In [ ]:
result = compare.distances(Z_ref, Z_query, n_sub=500, n_perm=1000)
for k, v in result.items():
    print(f'{k:>10}: {v}')

## 4. UMAP colored by group

In [ ]:
sc.pl.umap(adata, color='_group', title='ctrl vs IFN-β (monocytes)', frameon=False)

## 5. Localize — which clusters gained/lost query cells

Leiden-cluster the shared latent, then test each cluster's ctrl/stim composition
(Fisher exact + BH). Clusters flagged `gained`/`lost` are the localized shift.

In [ ]:
shift_table = localize.localize(adata, resolution=1.0)
shift_table[['cluster', 'n_ref', 'n_query', 'log2_enrichment', 'qval', 'direction']]

In [ ]:
sc.pl.umap(adata, color='_shift', title='Shifted clusters', frameon=False)

## 6. Drivers — Wilcoxon DE (should surface interferon-stimulated genes)

Expect ISG15, IFI6, IFIT1/3, MX1, OAS1, CXCL10 — the textbook type-I interferon response.

In [ ]:
drv = drivers.drivers(adata)
print('Up in stim  :', ', '.join(drv['up']['gene'].head(15)))
print('Down in stim:', ', '.join(drv['down']['gene'].head(10)))

## 7. Interpret — Claude turns the numbers into a cited rationale

Needs `ANTHROPIC_API_KEY`. We hand Claude only the structured evidence and log the
exact prompt + inputs, so every claim traces back to a number.

In [ ]:
# import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'   # set your key
evidence = interpret.assemble_evidence(adata, result, shift_table, drv)
record = interpret.interpret(evidence)
print(record['output'])

## 8. Same tool, every domain (this is the point)

Nothing above is specific to interferon or to monocytes. The identical three-line pipeline
runs across **immunology, host–pathogen disease, developmental biology, and cell-type ID** —
and recovers the *correct, distinct* biology each time. Same code, four different fields.

In [ ]:
def shiftscope(adata):
    Zr, Zq = embed.embed(adata, run_umap=False)
    d = compare.distances(Zr, Zq, n_sub=500, n_perm=500)
    up = drivers.drivers(adata)['up']['gene'].head(8).tolist()
    return d['edistance'], d['p_value'], up

cases = [
    ('immune — mouse phagocytes: unst vs LPS',   io.load_public('hagai', 'unst', 'LPS6', within=('species', 'mouse'))),
    ('disease — gut: healthy vs H. polygyrus',   io.load_public('haber', 'Control', 'Hpoly.Day10')),
    ('development — blood: monocyte vs erythroid', io.load_public('paul15', '14Mo', '2Ery')),
    ('cell types — pbmc68k: Monocyte vs B cell',  io.load_public('pbmc68k', 'CD14+ Monocyte', 'CD19+ B')),
]
for name, ad in cases:
    e, p, up = shiftscope(ad)
    print(f'{name:44s}  E-dist={e:8.1f}  p={p:.3f}  top: {", ".join(up[:6])}')
# Expect: LPS->Ccl3/4/Il1a (NF-κB); infection->Reg3b/g,Defa (antimicrobial);
#         erythroid->Klf1,Car2 (master TF); Monocyte->TYROBP,S100A8 (myeloid). Distinct each time.

## 8b. Rank many conditions by effect size

`compare.rank()` orders every condition by how far it moves the population — the core
Perturb-seq question, "which perturbation does the most?" On Dong 2023 it correctly ranks
IFNβ+IFNγ > IFNβ > IFNγ. On the Marson screen, the same call ranks every gene knockdown.

In [ ]:
# rank(): order many conditions by how far each shifts the population
# (the core Perturb-seq question — "which perturbation does the most?")
dong = io._preprocess_if_needed(io._fetch_public(io.PUBLIC_DATASETS['dong']))
compare.rank(dong, 'perturbation', 'No stimulation')  # IFNb+IFNg > IFNb > IFNg

**Rank a real CRISPR screen.** The same `rank()` call, on the Shifrut 2018 human CD8 T-cell
screen — 20 knockouts, ± TCR stimulation (Marson lab). It recovers the actual biology: the
TCR-signaling core (**LCP2/SLP-76, CD3D**) shifts cells most, followed by the **negative
regulators (RASA2, CBLB, SOCS1, TCEB2)** that paper found *enhance* T-cell function — the
same class of immunotherapy targets. A second real screen, no cell streaming.

In [ ]:
import numpy as np
# KO effects show under stimulation -> restrict to stim, annotated cells; rank all KOs vs control.
sh = io._fetch_public(io.PUBLIC_DATASETS['shifrut'])
sh = sh[(sh.obs['perturbation_2'].astype(str) == 'stim') & sh.obs['perturbation'].notna()].copy()
pert = sh.obs['perturbation'].astype(str).values
rng = np.random.default_rng(0)                        # cap 250 cells/KO for a fast live demo
keep = []
for g in np.unique(pert):
    gi = np.where(pert == g)[0]
    keep.append(gi if len(gi) <= 250 else rng.choice(gi, 250, replace=False))
sh = io._preprocess_if_needed(sh[np.sort(np.concatenate(keep))].copy())
compare.rank(sh, 'perturbation', 'control').head(10)  # LCP2/CD3D core + RASA2/CBLB/SOCS1 regulators

## 8c. Prioritize — which hits deserve a validation experiment?

Ranking hits by effect size isn't enough: the top of any screen is dominated by genes we
already understand. The decision a scientist actually faces is *which strong hit is worth
bench time* — and that means **strong phenotype × under-studied**.

`prioritize.py` scores both axes on the **real Marson CD4+ T-cell screen** (11k knockdowns,
`DE_stats.suppl_table.csv` — no cell streaming needed). The under-studied axis is
**grounded**, not guessed: a live NCBI PubMed query counts how many papers discuss each gene
in T-cell / immune biology. Then Claude scores novelty (1–5) and gives a validate/skip
verdict — with the real paper count in front of it, so every call traces back to data.

Watch the tool sink the textbook genes (**VAV1**, **CD45/PTPRC** — hundreds of papers) and
surface strong-but-obscure chromatin regulators (**TADA2B**, **SGF29**, **ELOF1** — 0–4
papers) as the "validate these" shortlist.

In [ ]:
from shiftscope import prioritize

# End-to-end: Marson hits -> live PubMed grounding -> Claude verdict -> ranked shortlist.
# Needs ANTHROPIC_API_KEY for the novelty/verdict columns; without it you still get the
# grounded phenotype + PubMed ranking and the figure.
shortlist = prioritize.prioritize(condition='Stim48hr', top_n=20)
shortlist[['gene', 'phenotype', 'pubmed_count', 'priority_score',
           'novelty', 'recommendation', 'verdict']]

In [ ]:
# The money figure: phenotype strength (y) vs literature coverage (x).
# Top-LEFT quadrant = strong + under-studied = the hits to validate.
prioritize.plot(shortlist);

## 8d. Calibration — when does it work? (operating boundaries)

A distance-with-a-p-value is only trustworthy if you know where it breaks. `calibration.py`
stress-tests the E-test on the *known* Kang IFN-β shift and maps two limits, reporting
**power** (fraction of random draws reaching p<0.05) over many repeats — not one lucky call:

- **vs cell number** — subsample down to 10 cells/group (on a deliberately subtle shift, so
  cell count is the limiting factor). Power collapses below ~100 cells/group.
- **vs effect size** — dilute the query with reference cells until the shift vanishes. It
  stays detectable until ~90% of the query is reference cells (the minimum detectable shift).

This is the honest "here's the envelope" figure — exactly the rigor a reviewer asks for.
Takes ~20 s.

In [ ]:
from shiftscope import calibration

# verbose=True prints the one-line envelope summary itself; loads Kang monocytes, runs both sweeps
cal = calibration.calibrate(reps=20, n_perm=200)
calibration.plot(cal['cells'], cal['effect']);

## 9. (Advanced) Genome-scale streaming — Marson CRISPRi Perturb-seq

`load_demo` streams a subsample straight from the 140 GB remote h5ad (GEO GSE314342) —
never downloading it whole. Fast on Colab; slow on a high-latency link. Uncomment to run
NTC vs a SETDB1 knockdown in resting CD4+ T cells.

In [ ]:
# adata_m = io.load_demo(query='SETDB1', condition='Rest', n_per_group=300)
# Zr, Zq = embed.embed(adata_m)
# print(compare.distances(Zr, Zq))